In [1]:
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# Настройка MLflow — логирование в локальную папку (можно заменить на сервер)
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("churn_experiments")

D:\Projects\4_Sem\AI_DPO\.venvv\Lib\site-packages\pydantic\_internal\_config.py:386: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)
2026/05/30 19:14:51 INFO mlflow.tracking.fluent: Experiment with name 'churn_experiments' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///D:/Projects/4_Sem/AI_DPO/project/notebooks/mlruns/957037974939759450', creation_time=1780157691211, experiment_id='957037974939759450', last_update_time=1780157691211, lifecycle_stage='active', name='churn_experiments', tags={}>

In [2]:
X_train = np.load('../artifacts/X_train_preprocessed.npy')
X_test = np.load('../artifacts/X_test_preprocessed.npy')
y_train = np.load('../artifacts/y_train.npy')
y_test = np.load('../artifacts/y_test.npy')
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Доля оттока train: {y_train.mean():.2%}, test: {y_test.mean():.2%}")

Train: (5634, 38), Test: (1409, 38)
Доля оттока train: 26.54%, test: 26.54%


In [9]:
def evaluate_and_log(model, model_name, params, X_train, y_train, X_test, y_test):
    with mlflow.start_run(run_name=model_name):
        # Логируем параметры
        mlflow.log_params(params)

        # Обучение
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        # Метрики
        metrics = {
            'model_name': model_name,                     # ← добавляем имя модели
            'auc_roc': roc_auc_score(y_test, y_proba),
            'f1': f1_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred),
            'recall': recall_score(y_test, y_pred)
        }
        # Логируем метрики (без model_name)
        mlflow.log_metrics({k: v for k, v in metrics.items() if k != 'model_name'})

        # Сохранение модели
        mlflow.sklearn.log_model(model, artifact_path="model")

        # Вывод
        print(f"{model_name}:")
        for k, v in metrics.items():
            if k != 'model_name':
                print(f"  {k}: {v:.4f}")

        return metrics, model

In [10]:
params_lr = {
    'max_iter': 1000,
    'random_state': 42
}
lr = LogisticRegression(**params_lr)
metrics_lr, model_lr = evaluate_and_log(lr, "LogisticRegression", params_lr,
                                         X_train, y_train, X_test, y_test)

LogisticRegression:
  auc_roc: 0.8428
  f1: 0.5872
  precision: 0.6633
  recall: 0.5267


In [11]:
params_rf = {
    'n_estimators': 100,
    'random_state': 42
}
rf = RandomForestClassifier(**params_rf)
metrics_rf, model_rf = evaluate_and_log(rf, "RandomForest", params_rf,
                                        X_train, y_train, X_test, y_test)

RandomForest:
  auc_roc: 0.8210
  f1: 0.5625
  precision: 0.6342
  recall: 0.5053


In [12]:
params_xgb_base = {
    'n_estimators': 100,
    'learning_rate': 0.1,
    'random_state': 42,
    'use_label_encoder': False,
    'eval_metric': 'logloss'
}
xgb_base = XGBClassifier(**params_xgb_base)
metrics_xgb, model_xgb = evaluate_and_log(xgb_base, "XGBoost_baseline", params_xgb_base,
                                          X_train, y_train, X_test, y_test)

XGBoost_baseline:
  auc_roc: 0.8425
  f1: 0.5808
  precision: 0.6246
  recall: 0.5428


In [13]:
params_xgb_tuned = {
    'n_estimators': 200,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'use_label_encoder': False,
    'eval_metric': 'logloss'
}
xgb_tuned = XGBClassifier(**params_xgb_tuned)
metrics_tuned, model_tuned = evaluate_and_log(xgb_tuned, "XGBoost_tuned", params_xgb_tuned,
                                              X_train, y_train, X_test, y_test)

XGBoost_tuned:
  auc_roc: 0.8409
  f1: 0.5874
  precision: 0.6515
  recall: 0.5348


In [14]:
import pandas as pd

results = pd.DataFrame([metrics_lr, metrics_rf, metrics_xgb, metrics_tuned])
print("Сводка метрик:")
print(results.round(4))

# Выбор лучшей модели по AUC-ROC
best_auc = max(m['auc_roc'] for m in [metrics_lr, metrics_rf, metrics_xgb, metrics_tuned])
best_metrics = [m for m in [metrics_lr, metrics_rf, metrics_xgb, metrics_tuned] if m['auc_roc'] == best_auc][0]
best_model_name = best_metrics['model_name']

print(f"\nЛучшая модель по AUC-ROC: {best_model_name} (AUC = {best_auc:.4f})")

Сводка метрик:
           model_name  auc_roc      f1  precision  recall
0  LogisticRegression   0.8428  0.5872     0.6633  0.5267
1        RandomForest   0.8210  0.5625     0.6342  0.5053
2    XGBoost_baseline   0.8425  0.5808     0.6246  0.5428
3       XGBoost_tuned   0.8409  0.5874     0.6515  0.5348

Лучшая модель по AUC-ROC: LogisticRegression (AUC = 0.8428)


In [15]:
# Определяем лучшую модель
if best_model_name == "LogisticRegression":
    final_model = model_lr
    final_params = params_lr
elif best_model_name == "RandomForest":
    final_model = model_rf
    final_params = params_rf
elif best_model_name == "XGBoost_baseline":
    final_model = model_xgb
    final_params = params_xgb_base
else:
    final_model = model_tuned
    final_params = params_xgb_tuned

# Логируем финальную модель отдельным run
with mlflow.start_run(run_name="Final_model"):
    mlflow.log_params(final_params)
    mlflow.log_metrics({k: metrics_lr[k] for k in ['auc_roc', 'f1', 'precision', 'recall']})
    mlflow.sklearn.log_model(final_model, artifact_path="final_model")
    # Сохраняем локально
    joblib.dump(final_model, '../artifacts/churn_model_final.joblib')
    print(f"Финальная модель ({best_model_name}) сохранена в artifacts/churn_model_final.joblib")
    print(f"AUC-ROC финальной модели: {best_auc:.4f}")

Финальная модель (LogisticRegression) сохранена в artifacts/churn_model_final.joblib
AUC-ROC финальной модели: 0.8428


In [18]:
print("Для просмотра экспериментов выполните в терминале:")
print("cd notebooks")
print("mlflow ui --backend-store-uri file:./mlruns")

Для просмотра экспериментов выполните в терминале:
cd notebooks
mlflow ui --backend-store-uri file:./mlruns
